<h1 style="font-size: 1.6rem; font-weight: bold">ITO 5217: Natural Language Processing</h1>
<h1 style="font-size: 1.6rem; font-weight: bold">Module 6.2: Transformer Architecture</h1>
<p style="margin-top: 5px; margin-bottom: 5px;">Monash University Australia</p>
<p style="margin-top: 5px; margin-bottom: 5px;">Jupyter Notebook by: Tristan Sim Yook Min</p>
References: Information Source from Monash Faculty of Information Technology

---

### **The Transformer Architecture**

> **Reference:** Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł., & Polosukhin, I. (2017). *Attention Is All You Need*. Advances in Neural Information Processing Systems (NeurIPS), 30. [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)

The Transformer is an encoder-decoder architecture that relies entirely on attention — no recurrence, no convolutions. Both the encoder and decoder are stacked 6 layers deep in the original paper.

<br>

#### **Encoder: Building it piece by piece**

Each encoder layer has two sub-layers, applied in order:

1. **Multi-Head Self-Attention**
2. **Feed-Forward Network**

Both are wrapped with a residual connection and layer normalization (details below).

<br>

#### **Step 1: Self-Attention (per word)**

For each word $x_i$, compute its query, key, and value:

$$q_i = W^Q x_i \quad \quad k_i = W^K x_i \quad \quad v_i = W^V x_i$$

Then compute the attention score between query $i$ and key $j$:

$$e_{ij} = q_i \cdot k_j$$

Normalize with softmax to get attention weights:

$$\alpha_{ij} = \text{softmax}(e_{ij}) = \frac{\exp(e_{ij})}{\sum_k \exp(e_{ik})}$$

Take a weighted sum of values to get the output for word $i$:

$$\text{Output}_i = \sum_j \alpha_{ij} v_j$$

<br>

#### **Step 1 (Vectorized): All words at once**

Stack all embeddings into $X$ and compute:

$$Q = XW^Q \quad \quad K = XW^K \quad \quad V = XW^V$$

$$E = QK^T$$

$$A = \text{softmax}(E)$$

$$\text{Output} = AV$$

Or in one line:

$$\text{Output} = \text{softmax}(QK^T) V$$

<br>

#### **Step 2: Feed-Forward Network**

Self-attention alone is just re-averaging value vectors — there are no non-linearities. Adding a feed-forward layer after attention introduces non-linear activation and additional expressive power.

$$m_i = \text{MLP}(\text{output}_i) = W_2 \cdot \text{ReLU}(W_1 \times \text{output}_i + b_1) + b_2$$

This is applied independently to each token position (same weights shared across positions).

<br>

#### **Training Trick #1: Residual Connections**

Deep networks struggle to learn the identity function. Residual connections let the raw input bypass each sub-layer and get added to the output:

$$x_\ell = F(x_{\ell-1}) + x_{\ell-1}$$

This prevents the network from "forgetting" important information as it passes through many layers. The gradient also flows more easily during backpropagation.

<br>

#### **Training Trick #2: Layer Normalization**

The input to each layer keeps shifting as the layers beneath update during training. Layer norm stabilizes this by normalizing each layer's activations to zero mean and unit variance:

$$\mu^\ell = \frac{1}{H} \sum_{i=1}^{H} a_i^\ell \quad \quad \sigma^\ell = \sqrt{\frac{1}{H} \sum_{i=1}^{H} (a_i^\ell - \mu^\ell)^2}$$

$$x^{\ell'} = \frac{x^\ell - \mu^\ell}{\sigma^\ell + \epsilon}$$

Where $H$ is the number of hidden units in the layer and $\epsilon$ is a small constant for numerical stability.

<br>

#### **Training Trick #3: Scaled Dot Product Attention**

After layer norm, activations have mean 0 and variance 1. But the dot product's variance scales with dimensionality $d_k$ — if $q$ and $k$ both have mean 0 and variance 1, their dot product has mean 0 but variance $d_k$. This pushes softmax into saturated regions with near-zero gradients.

Dividing by $\sqrt{d_k}$ brings the variance back to 1:

$$\text{Output} = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

<br>

#### **Step 3: Positional Encoding**

Self-attention treats the input as a *set* — it has no notion of word order. The sentence "dog bites man" and "man bites dog" would produce identical outputs without position information.

The fix: add a position vector $p_i \in \mathbb{R}^d$ to each token's embedding before feeding it into the model:

$$v_i = \tilde{v}_i + p_i \quad \quad q_i = \tilde{q}_i + p_i \quad \quad k_i = \tilde{k}_i + p_i$$

The original paper uses sinusoidal functions to generate $p_i$, so each position gets a unique, deterministic vector that the model can use to infer relative distances between tokens.

<br>

#### **Complete Encoder Layer**

Putting it all together, each of the 6 encoder layers does:

$$\text{Input} \xrightarrow{+ \text{positional encoding}} \text{Multi-Head Self-Attention} \xrightarrow{\text{Add \& Norm}} \text{Feed Forward} \xrightarrow{\text{Add \& Norm}} \text{Output}$$

<br>

---


### **The Decoder**

The decoder is similar to the encoder but has three sub-layers instead of two, and introduces masking to prevent looking at future tokens.

<br>

#### **Sub-layer 1: Masked Multi-Head Self-Attention**

**Problem:** During training, the decoder has access to the entire target sequence. Without masking, it could "cheat" by looking ahead at words it hasn't generated yet.

**Solution:** Set attention scores to $-\infty$ for all future positions before applying softmax. After softmax, these become 0 — effectively invisible.

$$e_{ij} = \begin{cases} q_i^T k_j, & j < i \\ -\infty, & j \geq i \end{cases}$$

This creates a lower-triangular attention matrix where each token can only attend to itself and previous tokens. Crucially, this still allows parallel training — all positions are computed simultaneously, but each sees only the past.

<br>

#### **Sub-layer 2: Encoder-Decoder Cross-Attention**

This is where the decoder "reads" the encoder's output. Unlike self-attention where Q, K, V all come from the same source:

- **Keys and Values** come from the encoder output: $k_i = Kh_i$, $v_i = Vh_i$
- **Queries** come from the decoder: $q_i = Qz_i$

Where:
- $h_1, \dots, h_T \in \mathbb{R}^d$ = output vectors from the encoder
- $z_1, \dots, z_T \in \mathbb{R}^d$ = input vectors from the decoder

This lets each decoder position attend to all positions in the input sequence — like the decoder asking "which parts of the input are relevant for generating this output word?"

<br>

#### **Sub-layer 3: Feed Forward**

Same as the encoder: a two-layer MLP with ReLU, applied independently per position.

<br>

#### **Final Output Layer**

After the last decoder layer:

1. **Linear layer** projects embeddings to a vector of length $|V|$ (vocabulary size)
2. **Softmax** converts this into a probability distribution over all possible next words

<br>

#### **Complete Decoder Layer**

$$\text{Output Embedding} \xrightarrow{+ \text{pos enc}} \text{Masked MH Self-Attn} \xrightarrow{\text{Add \& Norm}} \text{MH Cross-Attn} \xrightarrow{\text{Add \& Norm}} \text{FF} \xrightarrow{\text{Add \& Norm}} \text{Output}$$

<br>

#### **Transformer Drawbacks**

- **Quadratic computation:** Self-attention compares every pair of tokens, so cost grows as $O(n^2)$ with sequence length. RNNs grow linearly.
- **Position representations:** Are simple absolute position indices the best approach? This remains an active area of research (e.g., RoPE, ALiBi).

<br>

---


### **Transformer Language Models**

The Transformer architecture can be split into three families based on which components are used. The choice of architecture influences how the model is pretrained and what tasks it naturally fits.

<br>

#### **Architecture Family 1: Encoder-Decoders**

**Examples:** Original Transformer, T5, BART

The encoder processes the input bidirectionally (every token sees every other token), and the decoder generates output autoregressively (one token at a time, left to right).

**Pretraining approach (T5 — Span Corruption):** Replace random spans in the input with placeholder tokens, feed the corrupted text to the encoder, and train the decoder to reconstruct the missing spans.

**Example:**
- Original: "Thank you for inviting me to your party last week."
- Input: "Thank you `<X>` me to your party `<Y>` week."
- Target: "`<X>` for inviting `<Y>` last `<Z>`"

<br>

#### **Architecture Family 2: Decoders (Autoregressive LMs)**

**Examples:** GPT, GPT-2, GPT-3, LaMDA, PaLM

Decoder-only models are trained to predict the next word given all previous words:

$$P(x_t | x_{1..t-1})$$

They use masked self-attention so each position can only see past tokens. They're natural text generators but can't condition on future context.

| Model | Heads | Parameters | Layers | Hidden Dim |
|---|---|---|---|---|
| GPT | 12 | 117M | 12 | 768 |
| GPT-2 | 12 | 1.5B | 48 | 1600 |
| GPT-3 | 128 | 175B | 96 | 12288 |

**Finetuning GPT:** Append an `[EXTRACT]` token to the input and apply a linear classifier to its final representation. Different task formats (classification, entailment, similarity, multiple choice) are handled by arranging inputs with delimiter tokens.

**PaLM (2022):** A 540B parameter dense decoder-only Transformer. Showed that many benchmark tasks exhibited "discontinuous improvements" — performance jumped sharply only at the largest scale, suggesting some capabilities emerge only beyond certain model sizes.

<br>

#### **Architecture Family 3: Encoders**

**Examples:** BERT, RoBERTa

Encoder-only models see the full bidirectional context — every token can attend to every other token, including future ones. This is great for understanding tasks but makes them unsuited for generation.

**Pretraining approach (BERT — Masked Language Modeling):** Randomly mask 15% of tokens and train the model to predict them based on surrounding context:

$$P(x_t | x_{1..t-1, t+1..T})$$

The masking strategy: 80% replaced with `[MASK]`, 10% replaced with a random word, 10% kept unchanged. The model must predict the original word for all selected positions.

| Model | Heads | Parameters | Layers | Hidden Dim |
|---|---|---|---|---|
| BERT-Base | 12 | 110M | 12 | 768 |
| BERT-Large | 16 | 240M | 24 | 1024 |

<br>

#### **When to use which?**

- **Need to generate text?** → Use a decoder (GPT-style) or encoder-decoder (T5-style)
- **Need to understand/classify text?** → Use an encoder (BERT-style) — bidirectional context gives richer representations
- **Need both understanding and generation?** → Use an encoder-decoder (T5-style)

Encoders like BERT are not well-suited for autoregressive generation because they're designed to predict a single masked token, not to generate sequences one word at a time.

---


### **Pretraining and Finetuning**

**Goal:** Train a large model on massive amounts of unlabeled text first (pretraining), then adapt it to specific tasks with much smaller labeled datasets (finetuning).

Pretraining serves as a smart parameter initialization. Instead of starting from random weights, the model begins with a rich understanding of language — grammar, facts, reasoning patterns — learned from billions of words.

<br>

#### **The Two-Step Paradigm**

**Step 1: Pretrain (on language modeling)** Feed the model lots of text and let it learn general language patterns. No task-specific labels needed.

**Step 2:Finetune (on your task):** Take the pretrained model and adapt it to a specific downstream task (e.g., sentiment classification, question answering) using a smaller labeled dataset.

<br>

#### **Finetuning Strategies for BERT**

When adapting BERT for classification, we add a classifier head (feed-forward network + softmax) on top of the `[CLS]` token representation. The question is: how much of BERT do we update?

<br>

**Strategy 1: Finetune all layers**

Update every parameter in BERT along with the classifier. This gives the best performance but is computationally expensive, and you need a separate copy of the full model for each task.

$$\text{Input} \xrightarrow{[\text{CLS}] + \text{tokens}} \text{BERT (all layers trainable)} \xrightarrow{\varphi(x)} \text{Classifier} \xrightarrow{} \text{Prediction}$$

<br>

**Strategy 2: Freeze BERT entirely**

Keep all BERT parameters fixed and only train the classifier head. BERT acts purely as a feature extractor — it takes in raw text $x$ and outputs a fixed representation $\varphi(x)$.

This is fast and memory-efficient but may underperform since BERT's representations can't adapt to the specific task.

<br>

**Strategy 3: Freeze $k$ layers, finetune the rest**

A middle ground: freeze the lower layers (which tend to capture general linguistic features) and finetune only the upper layers (which capture more task-specific patterns) along with the classifier.

<br>

**Strategy 4:  Adapters**

> **Reference:** Houlsby, N., et al. (2019). *Parameter-Efficient Transfer Learning for NLP*. ICML.

Keep all of BERT frozen but insert small trainable **adapter modules** inside each transformer layer. Each adapter is a bottleneck architecture: down-project → nonlinearity → up-project, with a residual connection.

The key insight: adapters achieve nearly the same performance as full finetuning while training only ~3.6% of the total parameters.

| Method | Total Params | Trained Params / Task | GLUE Score |
|---|---|---|---|
| BERT-Large (full finetune) | 9.0x | 100% | 80.4 |
| Adapters (size 256) | 1.3x | 3.6% | 80.0 |
| Adapters (size 64) | 1.2x | 2.1% | 79.6 |

Adapters are especially useful when you need to support many tasks — instead of storing a separate copy of the full model for each task, you store one frozen base model and a small set of adapter weights per task.